# Sesión 3 · AC_PR_3_1 · Calibración de potenciómetros
**Sistemas de adquisición de datos · TSU Electrónica y Pruebas · Jabil · 2027**

Datos reales medidos por la generación 2025 con Arduino (10 bits, 5 V). No necesitas programar: ejecuta cada celda con ▶ en orden y cambia solo los campos del formulario.

In [ ]:
# @title Preparación (ejecuta esta celda primero; no hay nada que cambiar)
import numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (11, 4.2), "font.size": 12,
                     "axes.spines.top": False, "axes.spines.right": False})
# Mediciones de la generación 2025 (Arduino, convertidor de 10 bits, 5 V)
GRADOS = np.array([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200, 210, 220, 230, 240, 250, 260, 270, 280, 290])
SUBIDA = np.array([0.0, 0.0, 0.005, 0.22, 0.415, 0.606, 0.821, 1.007, 1.183, 1.393, 1.603, 1.808, 2.028, 2.278, 2.473, 2.669, 2.874, 3.079, 3.309, 3.524, 3.739, 3.93, 4.12, 4.301, 4.506, 4.663, 4.848, 5.0, 5.0, 5.0])
BAJADA = np.array([0, 0, 0.059, 0.244, 0.45, 0.65, 0.86, 1.017, 1.217, 1.403, 1.65, 1.818, 2.038, 2.258, 2.463, 2.654, 2.903, 3.113, 3.338, 3.529, 3.719, 3.935, 4.125, 4.326, 4.497, 4.687, 4.795, 5, 5, 5])
CM = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0])
V_DESL = np.array([0.0, 0.044, 0.225, 0.538, 1.408, 2.278, 2.947, 3.456, 3.978, 4.492, 4.814, 4.951, 5.0])
LECTURA_ROT = {"1": 0.6, "2": 0.98, "3": 1.36, "4": 1.74, "5": 2.12, "6": 2.5, "7": 2.88, "8": 3.26, "9": 3.64, "10": 4.02}
LECTURA_DESL = {"1": 0.5, "2": 0.92, "3": 1.34, "4": 1.76, "5": 2.18, "6": 2.6, "7": 3.02, "8": 3.44, "9": 3.86, "10": 4.28}

def ajusta(x, y, grado):
    c = np.polyfit(x, y, grado)
    yh = np.polyval(c, x)
    r2 = 1 - np.sum((y - yh) ** 2) / np.sum((y - y.mean()) ** 2)
    return c, yh, r2
print("Listo. Sigue con el bloque A.")


## Bloque A · El rotatorio
1. Mueve **desde** a 0 y **hasta** a 290, deja **recta**, ejecuta y anota R².
2. Mueve **desde** y **hasta** para quitar los tramos saturados. ¿Qué pasa con R² y con los residuos?
3. Termina con 30–260° y **recta**, y pasa m, b y R² a tu hoja.

In [ ]:
# @title Bloque A · Potenciómetro rotatorio: rango útil y modelo
desde = 30  # @param {type:"slider", min:0, max:140, step:10}
hasta = 260  # @param {type:"slider", min:150, max:290, step:10}
modelo = "recta"  # @param ["recta", "polinomio de grado 2", "polinomio de grado 3"]
grado = {"recta": 1, "polinomio de grado 2": 2, "polinomio de grado 3": 3}[modelo]
k = (GRADOS >= desde) & (GRADOS <= hasta)
x, y = GRADOS[k], SUBIDA[k]
c, yh, r2 = ajusta(x, y, grado)
CAL_ROT = dict(c=c, desde=desde, hasta=hasta, grado=grado)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.3))
a1.plot(GRADOS, SUBIDA, "o", color="#BBBBBB", label="fuera del rango elegido")
a1.plot(x, y, "o", color="#1F4E79", label="usados en el ajuste")
a1.plot(GRADOS, BAJADA, "s", ms=3, color="#D9822B", alpha=0.6, label="bajada (solo se muestra)")
xx = np.linspace(desde, hasta, 200); a1.plot(xx, np.polyval(c, xx), color="#D9822B", lw=2.5, label=modelo)
a1.set_xlabel("Ángulo [°]"); a1.set_ylabel("Salida [V]"); a1.legend(fontsize=9, frameon=False)
a2.axhline(0, color="#444", lw=1); a2.vlines(x, 0, y - yh, color="#C62828", lw=3)
a2.set_xlabel("Ángulo [°]"); a2.set_ylabel("Residuo [V]"); a2.set_title(f"Residuos · R² = {r2:.5f}")
plt.tight_layout(); plt.show()

print(f"Puntos usados: {k.sum()}   R² = {r2:.5f}   residuo máximo = {np.max(np.abs(y - yh)):.4f} V")
if grado == 1:
    print(f"m = {c[0]:.5f} V/°    b = {c[1]:.4f} V")
    print("Pega m, b y R² en la pestaña «Práctica», columna E, en los renglones «Pendiente m», «Ordenada b» y «R²»")
    print("(debajo de la tabla de datos; no en las filas 5 a 34).")
else:
    print("Coeficientes (de la mayor potencia a la constante):", np.round(c, 8))
plano = (np.diff(SUBIDA[k]) < 0.01).sum()
if plano:
    print(f"Ojo: incluiste {plano} tramo(s) donde la salida casi no cambia: el potenciómetro satura ahí.")


## Bloque B · El deslizable
Prueba grado 1 y grado 3. Mira la forma de los residuos, no solo R². Activa **ver_fuera** con grado 3 y con grado 6: ¿qué haría el modelo más allá de 6 cm?

In [ ]:
# @title Bloque B · Potenciómetro deslizable: ¿recta o polinomio?
grado_desl = 1  # @param {type:"slider", min:1, max:6, step:1}
ver_fuera = False  # @param {type:"boolean"}
c2, yh2, r2b = ajusta(CM, V_DESL, grado_desl)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.3))
a1.plot(CM, V_DESL, "o", color="#1F4E79", label="mediciones 2025")
x_max = 7.5 if ver_fuera else 6
xx = np.linspace(0, x_max, 300); a1.plot(xx, np.polyval(c2, xx), color="#D9822B", lw=2.5, label=f"grado {grado_desl}")
if ver_fuera:
    a1.axvspan(6, x_max, color="#F6D5D5", label="fuera de lo medido")
a1.set_xlabel("Posición [cm]"); a1.set_ylabel("Salida [V]"); a1.legend(fontsize=9, frameon=False)
a2.axhline(0, color="#444", lw=1); a2.vlines(CM, 0, V_DESL - yh2, color="#C62828", lw=3)
a2.set_xlabel("Posición [cm]"); a2.set_ylabel("Residuo [V]"); a2.set_title(f"Residuos · R² = {r2b:.4f}")
plt.tight_layout(); plt.show()
print(f"Grado {grado_desl}: R² = {r2b:.4f}, residuo máximo = {np.max(np.abs(V_DESL - yh2)):.3f} V")
print("Anota en tu hoja de entrega R² y el residuo máximo para grado 1 y grado 3, y si los residuos tienen patrón.")


## Bloque C · La lectura de tu equipo
Convierte las dos lecturas de tu equipo al valor físico y compáralas con la regla de tres.

In [ ]:
# @title Bloque C · Tu equipo: de la lectura al valor físico
equipo = 1  # @param {type:"slider", min:1, max:10, step:1}
V_rot, V_desl = LECTURA_ROT[str(equipo)], LECTURA_DESL[str(equipo)]
# Rotatorio: se invierte el modelo del bloque A dentro del rango calibrado
cA = CAL_ROT["c"]
if CAL_ROT["grado"] == 1:
    ang = (V_rot - cA[1]) / cA[0]
else:
    gg = np.linspace(CAL_ROT["desde"], CAL_ROT["hasta"], 20001)
    ang = gg[np.argmin(np.abs(np.polyval(cA, gg) - V_rot))]
dentro = CAL_ROT["desde"] <= ang <= CAL_ROT["hasta"]
# Deslizable: curva inversa (posición en función del voltaje), grado 3, de 0.5 a 5.5 cm
k = (CM >= 0.5) & (CM <= 5.5)
c_inv = np.polyfit(V_DESL[k], CM[k], 3)
pos = np.polyval(c_inv, V_desl)
print(f"Equipo {equipo}")
print(f"  Rotatorio:  {V_rot:.2f} V  →  {ang:.1f}°   ({'dentro' if dentro else 'FUERA'} del rango calibrado {CAL_ROT['desde']}–{CAL_ROT['hasta']}°)")
print(f"              regla de tres: {V_rot / 5 * 290:.1f}°")
print(f"  Deslizable: {V_desl:.2f} V  →  {pos:.2f} cm ± 0.2 cm  (regla de tres: {V_desl / 5 * 6:.2f} cm)")
if CAL_ROT["grado"] == 1:
    print("Pega el ángulo en la pestaña «Práctica», columna E, renglón «Ángulo de tu lectura».")
